# TMDB 5000 Movie Dataset으로 장르기반 추천시스템 만들기
* 장르에 있는 텍스트를 벡터화 후 텍스트 간 유사도를 구해 가까운 순서대로 정렬
* 유사도 척되 코사인 유사도

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib

In [2]:
data = pd.read_csv("https://raw.githubusercontent.com/haram4th/ablearn/main/tmdb_5000_movies.csv")
data.head()

,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2009-12-10,2787965087,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2007-05-19,961000000,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...","[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",2015-10-26,880674609,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 80, ""nam...",http://www.thedarkknightrises.com/,49026,"[{""id"": 849, ""name"": ""dc comics""}, {""id"": 853,...",en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,"[{""name"": ""Legendary Pictures"", ""id"": 923}, {""...","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-07-16,1084939099,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://movies.disney.com/john-carter,49529,"[{""id"": 818, ""name"": ""based on novel""}, {""id"":...",en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}]","[{""iso_3166_1"": ""US"", ""name"": ""United States o...",2012-03-07,284139100,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124


In [3]:
data = data.drop("homepage", axis=1)

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 19 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   id                    4803 non-null   int64  
 3   keywords              4803 non-null   object 
 4   original_language     4803 non-null   object 
 5   original_title        4803 non-null   object 
 6   overview              4800 non-null   object 
 7   popularity            4803 non-null   float64
 8   production_companies  4803 non-null   object 
 9   production_countries  4803 non-null   object 
 10  release_date          4802 non-null   object 
 11  revenue               4803 non-null   int64  
 12  runtime               4801 non-null   float64
 13  spoken_languages      4803 non-null   object 
 14  status                4803 non-null   object 
 15  tagline              

In [25]:
data.columns

Index(['budget', 'genres', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

In [26]:
dict_cols = ['genres', 'keywords', 'production_companies', 'production_countries', 'spoken_languages']

In [4]:
data['genres'][0]

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [7]:
import json

In [27]:
def dict_unpack(data):
    for col in dict_cols:
        temp = []
        for i in json.loads(data[col]):
            temp.append(i.get('name', 'None'))
        temp = " ".join(temp)
        data[col] = temp
    return data

In [28]:
data = data.apply(dict_unpack, axis=1)
data 

,budget,genres,id,keywords,original_language,original_title,overview,popularity,production_companies,production_countries,release_date,revenue,runtime,spoken_languages,status,tagline,title,vote_average,vote_count
0,237000000,Action Adventure Fantasy Science Fiction,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,Ingenious Film Partners Twentieth Century Fox ...,United States of America United Kingdom,2009-12-10,2787965087,162.0,English Español,Released,Enter the World of Pandora.,Avatar,7.2,11800
1,300000000,Adventure Fantasy Action,285,ocean drug abuse exotic island east india trad...,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,Walt Disney Pictures Jerry Bruckheimer Films S...,United States of America,2007-05-19,961000000,169.0,English,Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500
2,245000000,Action Adventure Crime,206647,spy based on novel secret agent sequel mi6 bri...,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,Columbia Pictures Danjaq B24,United Kingdom United States of America,2015-10-26,880674609,148.0,Français English Español Italiano Deutsch,Released,A Plan No One Escapes,Spectre,6.3,4466
3,250000000,Action Crime Drama Thriller,49026,dc comics crime fighter terrorist secret ident...,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,Legendary Pictures Warner Bros. DC Entertainme...,United States of America,2012-07-16,1084939099,165.0,English,Released,The Legend Ends,The Dark Knight Rises,7.6,9106
4,260000000,Action Adventure Science Fiction,49529,based on novel mars medallion space travel pri...,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,Walt Disney Pictures,United States of America,2012-03-07,284139100,132.0,English,Released,"Lost in our world, found in another.",John Carter,6.1,2124
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4798,220000,Action Crime Thriller,9367,united states–mexico barrier legs arms paper k...,es,El Mariachi,El Mariachi just wants to play his guitar and ...,14.269792,Columbia Pictures,Mexico United States of America,1992-09-04,2040920,81.0,Español,Released,"He didn't come looking for trouble, but troubl...",El Mariachi,6.6,238
4799,9000,Comedy Romance,72766,,en,Newlyweds,A newlywed couple's honeymoon is upended by th...,0.642552,,,2011-12-26,0,85.0,,Released,A newlywed couple's honeymoon is upended by th...,Newlyweds,5.9,5
4800,0,Comedy Drama Romance TV Movie,231617,date love at first sight narration investigati...,en,"Signed, Sealed, Delivered","""Signed, Sealed, Delivered"" introduces a dedic...",1.444476,Front Street Pictures Muse Entertainment Enter...,United States of America,2013-10-13,0,120.0,English,Released,NaN,"Signed, Sealed, Delivered",7.0,6
4801,0,,126186,,en,Shanghai Calling,When ambitious New York attorney Sam is sent t...,0.857008,,United States of America China,2012-05-03,0,98.0,English,Released,A New Yorker in Shanghai,Shanghai Calling,5.7,7


In [29]:
for col in dict_cols:
    print(data[col])

0       Action Adventure Fantasy Science Fiction
1                       Adventure Fantasy Action
2                         Action Adventure Crime
3                    Action Crime Drama Thriller
4               Action Adventure Science Fiction
                          ...                   
4798                       Action Crime Thriller
4799                              Comedy Romance
4800               Comedy Drama Romance TV Movie
4801                                            
4802                                 Documentary
Name: genres, Length: 4803, dtype: object
0       culture clash future space war space colony so...
1       ocean drug abuse exotic island east india trad...
2       spy based on novel secret agent sequel mi6 bri...
3       dc comics crime fighter terrorist secret ident...
4       based on novel mars medallion space travel pri...
                              ...                        
4798    united states–mexico barrier legs arms paper k...
4799         

In [8]:
result = ""
for item in json.loads(data['agenres'][0]):
    print(item['name'])
    result = result + " "

Action
Adventure
Fantasy
Science Fiction


In [10]:
data['spoken_languages'][0]

'[{"iso_639_1": "en", "name": "English"}, {"iso_639_1": "es", "name": "Espa\\u00f1ol"}]'

In [30]:
data['genres'].apply

<bound method Series.apply of 0       Action Adventure Fantasy Science Fiction
1                       Adventure Fantasy Action
2                         Action Adventure Crime
3                    Action Crime Drama Thriller
4               Action Adventure Science Fiction
                          ...                   
4798                       Action Crime Thriller
4799                              Comedy Romance
4800               Comedy Drama Romance TV Movie
4801                                            
4802                                 Documentary
Name: genres, Length: 4803, dtype: object>

In [31]:
data.columns

Index(['budget', 'genres', 'id', 'keywords', 'original_language',
       'original_title', 'overview', 'popularity', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'vote_average',
       'vote_count'],
      dtype='object')

In [32]:
data = data[[ 'original_title', 'genres', 'popularity', 'runtime', 'vote_average']]
data

,original_title,genres,popularity,runtime,vote_average
0,Avatar,Action Adventure Fantasy Science Fiction,150.437577,162.0,7.2
1,Pirates of the Caribbean: At World's End,Adventure Fantasy Action,139.082615,169.0,6.9
2,Spectre,Action Adventure Crime,107.376788,148.0,6.3
3,The Dark Knight Rises,Action Crime Drama Thriller,112.312950,165.0,7.6
4,John Carter,Action Adventure Science Fiction,43.926995,132.0,6.1
...,...,...,...,...,...
4798,El Mariachi,Action Crime Thriller,14.269792,81.0,6.6
4799,Newlyweds,Comedy Romance,0.642552,85.0,5.9
4800,"Signed, Sealed, Delivered",Comedy Drama Romance TV Movie,1.444476,120.0,7.0
4801,Shanghai Calling,,0.857008,98.0,5.7


In [34]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   original_title  4803 non-null   object 
 1   genres          4803 non-null   object 
 2   popularity      4803 non-null   float64
 3   runtime         4801 non-null   float64
 4   vote_average    4803 non-null   float64
dtypes: float64(3), object(2)
memory usage: 187.7+ KB


* 장르 컬럼의 텍스트를 숫자로 벡터화
* 코사인 유사도를 구해 비슷한 장르의 행 검색
* 유사도가 높은 행을 기준으로 내림차순 정렬

In [35]:
from sklearn.feature_extraction.text import CountVectorizer

In [36]:
cv = CountVectorizer(ngram_range=(1, 2))
genres_mat = cv.fit_transform(data['genres'])
genres_mat.shape
cv.get_feature_names_out()

array(['action', 'action adventure', 'action animation', 'action comedy',
       'action crime', 'action drama', 'action family', 'action fantasy',
       'action history', 'action horror', 'action mystery',
       'action romance', 'action science', 'action thriller',
       'action war', 'action western', 'adventure', 'adventure action',
       'adventure animation', 'adventure comedy', 'adventure crime',
       'adventure documentary', 'adventure drama', 'adventure family',
       'adventure fantasy', 'adventure history', 'adventure horror',
       'adventure mystery', 'adventure romance', 'adventure science',
       'adventure thriller', 'adventure war', 'adventure western',
       'animation', 'animation action', 'animation adventure',
       'animation comedy', 'animation drama', 'animation family',
       'animation fantasy', 'animation music', 'animation mystery',
       'animation science', 'animation war', 'comedy', 'comedy action',
       'comedy adventure', 'comedy animatio

In [40]:
genres_mat

<4803x276 sparse matrix of type '<class 'numpy.int64'>'
	with 20631 stored elements in Compressed Sparse Row format>

* 코사인 유사도를 이용해 genres_mat의 유사도 산출 후 genres_sim 생성
* 비슷한 장르의 영화를 찾기 위해 계산
* 코사인 유사도는 주로 문자, 문서의 유사도를 계산하는데 사용됨

In [41]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
n

In [51]:
genres_sim = cosine_similarity(genres_mat, genres_mat)
genres_sim.shape

(4803, 4803)

In [52]:
genres_sim_df = pd.DataFrame(genres_sim)
genres_sim_df

,0,1,2,3,4,5,6,7,8,9,...,4793,4794,4795,4796,4797,4798,4799,4800,4801,4802
0,1.000000,0.596285,0.447214,0.125988,0.755929,0.596285,0.0,0.755929,0.447214,0.745356,...,0.000000,0.000000,0.000000,0.377964,0.000000,0.149071,0.0000,0.000000,0.0,0.0
1,0.596285,1.000000,0.400000,0.169031,0.338062,0.800000,0.0,0.338062,0.600000,0.800000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.200000,0.0000,0.000000,0.0,0.0
2,0.447214,0.400000,1.000000,0.338062,0.507093,0.600000,0.0,0.507093,0.200000,0.600000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.400000,0.0000,0.000000,0.0,0.0
3,0.125988,0.169031,0.338062,1.000000,0.142857,0.169031,0.0,0.142857,0.000000,0.169031,...,0.377964,0.169031,0.377964,0.428571,0.218218,0.676123,0.0000,0.125988,0.0,0.0
4,0.755929,0.338062,0.507093,0.142857,1.000000,0.507093,0.0,1.000000,0.169031,0.507093,...,0.000000,0.000000,0.000000,0.428571,0.000000,0.169031,0.0000,0.000000,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4798,0.149071,0.200000,0.400000,0.676123,0.169031,0.200000,0.0,0.169031,0.000000,0.200000,...,0.000000,0.200000,0.000000,0.169031,0.258199,1.000000,0.0000,0.000000,0.0,0.0
4799,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.258199,0.000000,0.000000,0.000000,0.000000,1.0000,0.384900,0.0,0.0
4800,0.000000,0.000000,0.000000,0.125988,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.333333,0.149071,0.333333,0.125988,0.000000,0.000000,0.3849,1.000000,0.0,0.0
4801,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.000000,0.0,0.0


In [53]:
sorted_genre_sim = genres_sim.argsort()[:,::-1]

In [54]:
sorted_genre_sim[:1]

array([[   0, 3494,  813, ..., 3038, 3037, 2401]])

In [57]:
data.iloc[0, :]

original_title                                      Avatar
genres            Action Adventure Fantasy Science Fiction
popularity                                      150.437577
runtime                                              162.0
vote_average                                           7.2
Name: 0, dtype: object

In [58]:
data.iloc[14, :]

original_title                                Man of Steel
genres            Action Adventure Fantasy Science Fiction
popularity                                       99.398009
runtime                                              143.0
vote_average                                           6.5
Name: 14, dtype: object

In [59]:
data.iloc[870, :]

original_title                                 Superman II
genres            Action Adventure Fantasy Science Fiction
popularity                                       30.515175
runtime                                              127.0
vote_average                                           6.5
Name: 870, dtype: object

# 영화 이름과 추천 받을 개수를 입력받아 출력하는 함수

In [66]:
data[data['original_title'] == 'Avatar'].index.values[0]

0

In [63]:
sorted_genre_sim[0, 1:11]

array([3494,  813,  870,   46,   14, 1296, 1652,  419,  420, 1191])

In [69]:
data.iloc[sorted_genre_sim[0, 1:11], :]

,original_title,genres,popularity,runtime,vote_average
3494,Beastmaster 2: Through the Portal of Time,Action Adventure Fantasy Science Fiction,1.478505,107.0,4.6
813,Superman,Action Adventure Fantasy Science Fiction,48.507081,143.0,6.9
870,Superman II,Action Adventure Fantasy Science Fiction,30.515175,127.0,6.5
46,X-Men: Days of Future Past,Action Adventure Fantasy Science Fiction,118.078691,131.0,7.5
14,Man of Steel,Action Adventure Fantasy Science Fiction,99.398009,143.0,6.5
1296,Superman III,Comedy Action Adventure Fantasy Science Fiction,22.164202,125.0,5.3
1652,Dragonball Evolution,Action Adventure Fantasy Science Fiction Thriller,21.677732,85.0,2.9
419,Jumper,Adventure Fantasy Science Fiction,21.218000,88.0,5.9
420,Hellboy II: The Golden Army,Adventure Fantasy Science Fiction,58.579760,120.0,6.5
1191,Small Soldiers,Comedy Adventure Fantasy Science Fiction Action,23.088571,110.0,6.2


In [70]:
def find_sim_movie(data, sorted_genres_sim, title_name, top_n=10):
    title_movie = data[data['original_title'] == title_name]
    title_index = title_movie.index.values[0]
    similar_idxs = sorted_genre_sim[title_index, 1:top_n+1]
    return data.iloc[similar_idxs]

In [72]:
find_sim_movie(data, sorted_genre_sim, 'Man of Steel', 20)

,original_title,genres,popularity,runtime,vote_average
3494,Beastmaster 2: Through the Portal of Time,Action Adventure Fantasy Science Fiction,1.478505,107.0,4.6
813,Superman,Action Adventure Fantasy Science Fiction,48.507081,143.0,6.9
870,Superman II,Action Adventure Fantasy Science Fiction,30.515175,127.0,6.5
46,X-Men: Days of Future Past,Action Adventure Fantasy Science Fiction,118.078691,131.0,7.5
14,Man of Steel,Action Adventure Fantasy Science Fiction,99.398009,143.0,6.5
1296,Superman III,Comedy Action Adventure Fantasy Science Fiction,22.164202,125.0,5.3
1652,Dragonball Evolution,Action Adventure Fantasy Science Fiction Thriller,21.677732,85.0,2.9
419,Jumper,Adventure Fantasy Science Fiction,21.218000,88.0,5.9
420,Hellboy II: The Golden Army,Adventure Fantasy Science Fiction,58.579760,120.0,6.5
1191,Small Soldiers,Comedy Adventure Fantasy Science Fiction Action,23.088571,110.0,6.2
